## Model accuracy
Import libraries

In [1]:
import os
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.constants import Avogadro as UNA

Define inputs

In [2]:
model_name = "meno"
# Data indices
indices = {
  "x": [list(range(4)),[4],[5],[6]],
  "y": [list(range(7,11)), [11]]
}
# Paths
path_to_datagen = "./../datagen/"
paths = {
  "data": path_to_datagen + "/data/test/",
  "model": "./../03.merged/case.r06/saving/plasma/SavedModel/",
  "saving": "./output/error/"
}
os.makedirs(paths["saving"], exist_ok=True)
# Plotting
labels = {
  "Ar": r"Ar",
  "Ar4s": r"Ar$^{4s}$",
  "Ar4p": r"Ar$^{4p}$",
  "Ar+": r"Ar$^+$",
  "e-": r"e$^-$"
}
# Error
eps = 1e-10

In [3]:
# Time grid
tlim = [1e-14, 1e-7]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])

Set variables names

In [ ]:
names = np.array(list(data.values())[0].columns)
names = np.concatenate([names[i] for i in indices["y"]]).tolist()
names = [name[2:] for name in names]
names

Compute error

In [6]:
error = {"l2": [], "mape": []}
for idata in data.values():
  _, y_true, y_pred = utils.predict(model, idata.to_numpy(), indices, t)
  y_true /= UNA
  y_pred /= UNA
  error["l2"].append(utils.error_l2(y_true, y_pred, eps=eps))
  error["mape"].append(utils.error_mape(y_true, y_pred, eps=eps))

In [7]:
for (k, err) in error.items():
  kerr = np.vstack(err)
  kerr = np.vstack([np.mean(kerr, axis=0), np.std(kerr, axis=0)])
  kerr = pd.DataFrame(kerr, columns=names, index=["mean", "std"])
  kerr = kerr.sort_values(by=["mean"], axis=1)
  kerr.to_csv(paths["saving"]+f"/{model_name}_{k}.csv", float_format="%.3f")
  utils.write_latex(kerr, labels, paths["saving"]+f"/{model_name}_{k}_latex.txt")